In [1]:
#00_preprocessing.ipynb

In [2]:
# Imports and paths

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("../..")   # this is for local use, change if needed
RAW = BASE / "data" / "raw" / "CY-Bench" / "wheat" / "IN"
PROCESSED = BASE / "data" / "processed" / "CY-Bench"
PROCESSED.mkdir(parents=True, exist_ok=True)

CROP = "wheat"
COUNTRY = "IN"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [3]:
# File map

FILES = {
    "calendar": RAW / "crop_calendar_wheat_IN.csv",
    "mask": RAW / "crop_mask_wheat_IN.csv",
    "fpar": RAW / "fpar_wheat_IN.csv",
    "location": RAW / "location_wheat_IN.csv",
    "meteo": RAW / "meteo_wheat_IN.csv",
    "ndvi": RAW / "ndvi_wheat_IN.csv",
    "soil_moisture": RAW / "soil_moisture_wheat_IN.csv",
    "soil": RAW / "soil_wheat_IN.csv",
    "yield": RAW / "yield_wheat_IN.csv",
}

FILES

{'calendar': PosixPath('../../data/raw/CY-Bench/wheat/IN/crop_calendar_wheat_IN.csv'),
 'mask': PosixPath('../../data/raw/CY-Bench/wheat/IN/crop_mask_wheat_IN.csv'),
 'fpar': PosixPath('../../data/raw/CY-Bench/wheat/IN/fpar_wheat_IN.csv'),
 'location': PosixPath('../../data/raw/CY-Bench/wheat/IN/location_wheat_IN.csv'),
 'meteo': PosixPath('../../data/raw/CY-Bench/wheat/IN/meteo_wheat_IN.csv'),
 'ndvi': PosixPath('../../data/raw/CY-Bench/wheat/IN/ndvi_wheat_IN.csv'),
 'soil_moisture': PosixPath('../../data/raw/CY-Bench/wheat/IN/soil_moisture_wheat_IN.csv'),
 'soil': PosixPath('../../data/raw/CY-Bench/wheat/IN/soil_wheat_IN.csv'),
 'yield': PosixPath('../../data/raw/CY-Bench/wheat/IN/yield_wheat_IN.csv')}

In [4]:
# Loading data

dfs = {name: pd.read_csv(path) for name, path in FILES.items()}
{name: df.shape for name, df in dfs.items()}

{'calendar': (574, 4),
 'mask': (585, 4),
 'fpar': (475272, 4),
 'location': (574, 5),
 'meteo': (4821600, 11),
 'ndvi': (530395, 4),
 'soil_moisture': (4384212, 5),
 'soil': (574, 5),
 'yield': (11706, 12)}

In [5]:
# Schema inspection

schema = []
for name, df in dfs.items():
    for col in df.columns:
        schema.append({
            "table": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "missing_fraction": df[col].isna().mean(),
            "n_unique": df[col].nunique(dropna=True)
        })

schema_df = pd.DataFrame(schema).sort_values(["table", "column"]).reset_index(drop=True)
schema_df

,table,column,dtype,missing_fraction,n_unique
0,calendar,adm_id,str,0.0,574
1,calendar,crop_name,str,0.0,1
2,calendar,eos,float64,0.0,559
3,calendar,sos,float64,0.0,555
4,fpar,adm_id,str,0.0,574
5,fpar,crop_name,str,0.0,1
6,fpar,date,int64,0.0,828
7,fpar,fpar,float64,0.0,67774
8,location,adm_id,str,0.0,574
9,location,crop_name,str,0.0,1


In [6]:
# Quick preview of columns per table

for name, df in dfs.items():
    print(f"\n{name.upper()}  shape={df.shape}")
    print(df.columns.tolist())


CALENDAR  shape=(574, 4)
['crop_name', 'adm_id', 'sos', 'eos']

MASK  shape=(585, 4)
['crop_name', 'adm_id', 'crop_area', 'crop_area_percentage']

FPAR  shape=(475272, 4)
['crop_name', 'adm_id', 'date', 'fpar']

LOCATION  shape=(574, 5)
['crop_name', 'adm_id', 'latitude', 'longitude', 'region_area']

METEO  shape=(4821600, 11)
['crop_name', 'adm_id', 'date', 'tmin', 'tmax', 'prec', 'rad', 'tavg', 'et0', 'vpd', 'cwb']

NDVI  shape=(530395, 4)
['crop_name', 'adm_id', 'date', 'ndvi']

SOIL_MOISTURE  shape=(4384212, 5)
['crop_name', 'adm_id', 'date', 'ssm', 'rsm']

SOIL  shape=(574, 5)
['crop_name', 'adm_id', 'awc', 'bulk_density', 'drainage_class']

YIELD  shape=(11706, 12)
['crop_name', 'country_code', 'adm_id', 'season_name', 'planting_year', 'planting_date', 'harvest_year', 'harvest_date', 'yield', 'production', 'planted_area', 'harvest_area']


In [7]:
# Standardize key columns and dates

def standardize_keys(df):
    df = df.copy()
    if "crop_name" in df.columns:
        df["crop_name"] = df["crop_name"].astype(str).str.strip().str.lower()
    if "adm_id" in df.columns:
        df["adm_id"] = df["adm_id"].astype(str).str.strip()
    if "country_code" in df.columns:
        df["country_code"] = df["country_code"].astype(str).str.strip().str.upper()
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")
    return df

dfs = {name: standardize_keys(df) for name, df in dfs.items()}

In [8]:
# Keep only useful columns in yield and drop empty ones

yield_cols_keep = [
    "crop_name", "country_code", "adm_id", "harvest_year",
    "yield", "production", "planted", "harvest_area"
]

yield_df = dfs["yield"].copy()

existing_keep = [c for c in yield_cols_keep if c in yield_df.columns]
yield_df = yield_df[existing_keep].copy()

yield_df["harvest_year"] = pd.to_numeric(yield_df["harvest_year"], errors="coerce").astype("Int64")

for c in ["yield", "production", "planted", "harvest_area"]:
    if c in yield_df.columns:
        yield_df[c] = pd.to_numeric(yield_df[c], errors="coerce")

yield_df.head()

,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area
0,wheat,IN,IN-14-0001,1990,0.736,13400.0,18200.0
1,wheat,IN,IN-14-0001,1991,0.645,11800.0,18300.0
2,wheat,IN,IN-14-0001,1992,0.626,10700.0,17100.0
3,wheat,IN,IN-14-0001,1993,0.712,12100.0,17000.0
4,wheat,IN,IN-14-0001,1994,0.811,14200.0,17500.0


In [9]:
# Static tables

calendar_df = dfs["calendar"][["crop_name", "adm_id", "sos", "eos"]].copy()
mask_df = dfs["mask"][["crop_name", "adm_id", "crop_area", "crop_area_percentage"]].copy()
location_df = dfs["location"][["crop_name", "adm_id", "latitude", "longitude", "region_area"]].copy()
soil_df = dfs["soil"][["crop_name", "adm_id", "awc", "bulk_density", "drainage_class"]].copy()

for df_ in [calendar_df, mask_df, location_df, soil_df]:
    for c in df_.columns:
        if c not in ["crop_name", "adm_id", "drainage_class"]:
            df_[c] = pd.to_numeric(df_[c], errors="coerce")

In [10]:
# Time-series helper: assign harvest_year from calendar year for first-pass full-year panel

def add_harvest_year_from_date(df):
    df = df.copy()
    df = df[df["date"].notna()].copy()
    df["harvest_year"] = df["date"].dt.year.astype("Int64")
    return df

In [11]:
# Aggregate meteo to yearly summaries

meteo_df = add_harvest_year_from_date(dfs["meteo"])

meteo_agg = (
    meteo_df
    .groupby(["crop_name", "adm_id", "harvest_year"], as_index=False)
    .agg(
        avg_tmin=("tmin", "mean"),
        avg_tmax=("tmax", "mean"),
        avg_tavg=("tavg", "mean"),
        avg_prec=("prec", "mean"),
        sum_prec=("prec", "sum"),
        avg_rad=("rad", "mean"),
        avg_et0=("et0", "mean"),
        avg_vpd=("vpd", "mean"),
        avg_cwb=("cwb", "mean"),
        n_meteo_obs=("date", "count"),
    )
)

meteo_agg.head()

,crop_name,adm_id,harvest_year,avg_tmin,avg_tmax,avg_tavg,avg_prec,sum_prec,avg_rad,avg_et0,avg_vpd,avg_cwb,n_meteo_obs
0,wheat,IN-01-0044,2001,22.923170,31.562326,26.546184,4.218219,1539.650,1.777430e+07,4.257921,21.454299,-0.039701,365
1,wheat,IN-01-0044,2002,22.955153,32.061227,26.782847,2.409723,879.549,1.860318e+07,4.518570,23.201586,-2.108847,365
2,wheat,IN-01-0044,2003,23.057230,31.564753,26.613036,3.817392,1393.348,1.779816e+07,4.329186,21.621521,-0.511795,365
3,wheat,IN-01-0044,2004,22.784404,31.861710,26.584298,3.101787,1135.254,1.849911e+07,4.453418,22.929716,-1.351631,366
4,wheat,IN-01-0044,2005,23.114748,31.775299,26.746997,3.954614,1443.434,1.800680e+07,4.365512,22.036858,-0.410899,365


In [12]:
# Aggregate soil moisture to yearly summaries

sm_df = add_harvest_year_from_date(dfs["soil_moisture"])

sm_agg = (
    sm_df
    .groupby(["crop_name", "adm_id", "harvest_year"], as_index=False)
    .agg(
        avg_ssm=("ssm", "mean"),
        min_ssm=("ssm", "min"),
        max_ssm=("ssm", "max"),
        avg_rsm=("rsm", "mean"),
        min_rsm=("rsm", "min"),
        max_rsm=("rsm", "max"),
        n_sm_obs=("date", "count"),
    )
)

sm_agg.head()

,crop_name,adm_id,harvest_year,avg_ssm,min_ssm,max_ssm,avg_rsm,min_rsm,max_rsm,n_sm_obs
0,wheat,IN-01-0044,2003,4.075138,2.184,6.980,196.424308,154.910,292.577,334
1,wheat,IN-01-0044,2004,3.886773,2.422,6.142,187.242311,160.270,236.946,366
2,wheat,IN-01-0044,2005,3.845425,2.102,6.199,187.951353,157.975,262.617,365
3,wheat,IN-01-0044,2006,4.055208,2.795,6.878,194.639696,162.405,273.133,365
4,wheat,IN-01-0044,2007,4.233885,2.967,6.681,201.290197,163.515,279.360,365


In [13]:
# Aggregate NDVI to yearly summaries

ndvi_df = add_harvest_year_from_date(dfs["ndvi"])

ndvi_agg = (
    ndvi_df
    .groupby(["crop_name", "adm_id", "harvest_year"], as_index=False)
    .agg(
        avg_ndvi=("ndvi", "mean"),
        min_ndvi=("ndvi", "min"),
        max_ndvi=("ndvi", "max"),
        std_ndvi=("ndvi", "std"),
        n_ndvi_obs=("date", "count"),
    )
)

ndvi_agg.head()

,crop_name,adm_id,harvest_year,avg_ndvi,min_ndvi,max_ndvi,std_ndvi,n_ndvi_obs
0,wheat,IN-01-0044,2001,0.470875,0.291,0.811,0.142947,40
1,wheat,IN-01-0044,2002,0.478256,0.290,0.748,0.117053,43
2,wheat,IN-01-0044,2003,0.454717,0.263,0.780,0.145621,46
3,wheat,IN-01-0044,2004,0.522523,0.358,0.795,0.119371,44
4,wheat,IN-01-0044,2005,0.491400,0.304,0.796,0.142268,45


In [14]:
# Aggregate FPAR to yearly summaries

fpar_df = add_harvest_year_from_date(dfs["fpar"])

fpar_agg = (
    fpar_df
    .groupby(["crop_name", "adm_id", "harvest_year"], as_index=False)
    .agg(
        avg_fpar=("fpar", "mean"),
        min_fpar=("fpar", "min"),
        max_fpar=("fpar", "max"),
        std_fpar=("fpar", "std"),
        n_fpar_obs=("date", "count"),
    )
)

fpar_agg.head()

,crop_name,adm_id,harvest_year,avg_fpar,min_fpar,max_fpar,std_fpar,n_fpar_obs
0,wheat,IN-01-0044,2001,36.545861,23.948,53.044,11.071747,36
1,wheat,IN-01-0044,2002,38.357389,25.249,49.035,8.073604,36
2,wheat,IN-01-0044,2003,35.768833,20.149,55.791,11.662846,36
3,wheat,IN-01-0044,2004,40.567639,29.674,56.479,8.777414,36
4,wheat,IN-01-0044,2005,36.287889,25.645,48.853,8.422622,36


In [15]:
# Aggregate FPAR to yearly summaries

fpar_df = add_harvest_year_from_date(dfs["fpar"])

fpar_agg = (
    fpar_df
    .groupby(["crop_name", "adm_id", "harvest_year"], as_index=False)
    .agg(
        avg_fpar=("fpar", "mean"),
        min_fpar=("fpar", "min"),
        max_fpar=("fpar", "max"),
        std_fpar=("fpar", "std"),
        n_fpar_obs=("date", "count"),
    )
)

fpar_agg.head()

,crop_name,adm_id,harvest_year,avg_fpar,min_fpar,max_fpar,std_fpar,n_fpar_obs
0,wheat,IN-01-0044,2001,36.545861,23.948,53.044,11.071747,36
1,wheat,IN-01-0044,2002,38.357389,25.249,49.035,8.073604,36
2,wheat,IN-01-0044,2003,35.768833,20.149,55.791,11.662846,36
3,wheat,IN-01-0044,2004,40.567639,29.674,56.479,8.777414,36
4,wheat,IN-01-0044,2005,36.287889,25.645,48.853,8.422622,36


In [16]:
# Build the base panel from yield and static tables

panel = yield_df.copy()

for static_df, name in [
    (calendar_df, "calendar"),
    (mask_df, "mask"),
    (location_df, "location"),
    (soil_df, "soil"),
]:
    before = len(panel)
    panel = panel.merge(static_df, on=["crop_name", "adm_id"], how="left")
    after = len(panel)
    print(f"{name}: rows before={before}, after={after}")
    
panel.head()

calendar: rows before=11706, after=11706
mask: rows before=11706, after=11706
location: rows before=11706, after=11706
soil: rows before=11706, after=11706


,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area,sos,eos,crop_area,crop_area_percentage,latitude,longitude,region_area,awc,bulk_density,drainage_class
0,wheat,IN,IN-14-0001,1990,0.736,13400.0,18200.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0
1,wheat,IN,IN-14-0001,1991,0.645,11800.0,18300.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0
2,wheat,IN,IN-14-0001,1992,0.626,10700.0,17100.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0
3,wheat,IN,IN-14-0001,1993,0.712,12100.0,17000.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0
4,wheat,IN,IN-14-0001,1994,0.811,14200.0,17500.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0


In [17]:
# Merge yearly aggregated time-series tables

for yearly_df, name in [
    (meteo_agg, "meteo"),
    (sm_agg, "soil_moisture"),
    (ndvi_agg, "ndvi"),
    (fpar_agg, "fpar"),
]:
    before = len(panel)
    panel = panel.merge(yearly_df, on=["crop_name", "adm_id", "harvest_year"], how="left")
    after = len(panel)
    matched = panel.filter(regex=f"^n_").notna().any(axis=1).mean()
    print(f"{name}: rows before={before}, after={after}, any_ts_match_share={matched:.3f}")

panel.head()

meteo: rows before=11706, after=11706, any_ts_match_share=0.645
soil_moisture: rows before=11706, after=11706, any_ts_match_share=0.645
ndvi: rows before=11706, after=11706, any_ts_match_share=0.645
fpar: rows before=11706, after=11706, any_ts_match_share=0.645


,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area,sos,eos,crop_area,crop_area_percentage,latitude,longitude,region_area,awc,bulk_density,drainage_class,avg_tmin,avg_tmax,avg_tavg,avg_prec,sum_prec,avg_rad,avg_et0,avg_vpd,avg_cwb,n_meteo_obs,avg_ssm,min_ssm,max_ssm,avg_rsm,min_rsm,max_rsm,n_sm_obs,avg_ndvi,min_ndvi,max_ndvi,std_ndvi,n_ndvi_obs,avg_fpar,min_fpar,max_fpar,std_fpar,n_fpar_obs
0,wheat,IN,IN-14-0001,1990,0.736,13400.0,18200.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,wheat,IN,IN-14-0001,1991,0.645,11800.0,18300.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,wheat,IN,IN-14-0001,1992,0.626,10700.0,17100.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,wheat,IN,IN-14-0001,1993,0.712,12100.0,17000.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,wheat,IN,IN-14-0001,1994,0.811,14200.0,17500.0,339.086,101.91,93.6,4.076,21.45938,81.25028,2296.0,10.71,1.591,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Restrict to common usable data:
# keep rows where outcome exists and at least one yearly predictor block exists

predictor_count_cols = ["n_meteo_obs", "n_sm_obs", "n_ndvi_obs", "n_fpar_obs"]
existing_count_cols = [c for c in predictor_count_cols if c in panel.columns]

panel["has_any_ts"] = panel[existing_count_cols].notna().any(axis=1)
panel["n_available_blocks"] = panel[existing_count_cols].notna().sum(axis=1)

panel = panel[panel["yield"].notna()].copy()
panel_common = panel[panel["has_any_ts"]].copy()

panel.shape, panel_common.shape

((11706, 46), (7552, 46))

In [19]:
# Optional stricter version:
# keep only rows that have all four main yearly predictor groups

required_blocks = ["n_meteo_obs", "n_sm_obs", "n_ndvi_obs", "n_fpar_obs"]
required_blocks = [c for c in required_blocks if c in panel_common.columns]

panel_strict = panel_common.dropna(subset=required_blocks).copy()
panel_strict.shape

(6715, 46)

In [20]:
# Diagnostics: overlap and missingness

diagnostics = pd.DataFrame({
    "dataset": ["yield_base", "panel_after_merge", "panel_common", "panel_strict"],
    "n_rows": [len(yield_df), len(panel), len(panel_common), len(panel_strict)],
    "n_adm_ids": [
        yield_df["adm_id"].nunique(),
        panel["adm_id"].nunique(),
        panel_common["adm_id"].nunique(),
        panel_strict["adm_id"].nunique(),
    ],
    "year_min": [
        yield_df["harvest_year"].min(),
        panel["harvest_year"].min(),
        panel_common["harvest_year"].min(),
        panel_strict["harvest_year"].min(),
    ],
    "year_max": [
        yield_df["harvest_year"].max(),
        panel["harvest_year"].max(),
        panel_common["harvest_year"].max(),
        panel_strict["harvest_year"].max(),
    ],
})

diagnostics

,dataset,n_rows,n_adm_ids,year_min,year_max
0,yield_base,11706,520,1990,2017
1,panel_after_merge,11706,520,1990,2017
2,panel_common,7552,502,2001,2017
3,panel_strict,6715,501,2003,2017


In [21]:
# Missingness report for the recommended panel

missing_report = (
    panel_common
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_report.head(50)

,column,missing_fraction
0,max_rsm,0.110832
1,n_sm_obs,0.110832
2,avg_rsm,0.110832
3,min_ssm,0.110832
4,avg_ssm,0.110832
5,max_ssm,0.110832
6,min_rsm,0.110832
7,harvest_area,0.000000
8,crop_name,0.000000
9,country_code,0.000000


In [22]:
# Final cleanup for recommended panel

panel_common = panel_common.drop(columns=["has_any_ts"], errors="ignore").copy()

sort_cols = [c for c in ["crop_name", "adm_id", "harvest_year"] if c in panel_common.columns]
panel_common = panel_common.sort_values(sort_cols).reset_index(drop=True)

panel_common.head()

,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area,sos,eos,crop_area,crop_area_percentage,latitude,longitude,region_area,awc,bulk_density,drainage_class,avg_tmin,avg_tmax,avg_tavg,avg_prec,sum_prec,avg_rad,avg_et0,avg_vpd,avg_cwb,n_meteo_obs,avg_ssm,min_ssm,max_ssm,avg_rsm,min_rsm,max_rsm,n_sm_obs,avg_ndvi,min_ndvi,max_ndvi,std_ndvi,n_ndvi_obs,avg_fpar,min_fpar,max_fpar,std_fpar,n_fpar_obs,n_available_blocks
0,wheat,IN,IN-01-0047,2017,1.000,10.0,10.0,320.213,92.669,20.2,0.243,17.52128,81.18046,8316.0,14.203,1.506,4.0,23.450277,33.001384,27.813666,3.662170,1336.692,1.838194e+07,4.500485,26.257896,-0.838315,365.0,4.605701,2.041,8.040,256.749784,235.919,319.130,365.0,0.587410,0.374,0.758,0.128424,39.0,44.094639,24.974,62.886,13.939727,36.0,4
1,wheat,IN,IN-01-0049,2010,1.500,60.0,40.0,284.497,87.670,72.4,0.635,16.39034,79.72043,11398.0,15.192,1.542,4.0,23.812932,32.932452,27.655723,3.923219,1431.975,1.803340e+07,4.806238,26.328907,-0.883019,365.0,4.711455,1.947,8.433,259.160753,233.007,317.711,365.0,0.546682,0.284,0.761,0.137065,44.0,47.217833,27.859,71.492,14.510811,36.0,4
2,wheat,IN,IN-01-0051,2001,0.698,880.0,1260.0,313.989,102.880,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,22.820247,32.929427,27.616770,1.545647,564.161,1.922248e+07,5.392036,30.792145,-3.846389,365.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.364452,0.190,0.615,0.108494,42.0,28.438778,12.659,50.322,12.020800,36.0,3
3,wheat,IN,IN-01-0051,2002,1.000,1000.0,1000.0,313.989,102.880,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,22.865921,33.240953,27.804756,1.303027,475.605,1.973802e+07,5.441342,31.526392,-4.138315,365.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.386744,0.247,0.522,0.088545,43.0,31.348917,18.032,50.017,11.102711,36.0,3
4,wheat,IN,IN-01-0051,2003,0.304,210.0,690.0,313.989,102.880,186.8,1.072,15.56859,77.82124,17419.0,15.396,1.594,4.0,23.324778,33.771510,28.309710,1.217356,444.335,1.922673e+07,5.597868,33.526879,-4.380512,365.0,3.771593,0.827,7.338,277.465955,270.581,312.943,334.0,0.366091,0.193,0.595,0.121144,44.0,28.397778,13.847,53.197,13.007601,36.0,4


In [23]:
# Save outputs

panel_common_path = PROCESSED / "wheat_IN_panel_full_year.csv"
panel_strict_path = PROCESSED / "wheat_IN_panel_strict.csv"
schema_path = PROCESSED / "wheat_IN_schema_report.csv"
missing_path = PROCESSED / "wheat_IN_missing_report.csv"
diagnostics_path = PROCESSED / "wheat_IN_panel_diagnostics.csv"

panel_common.to_csv(panel_common_path, index=False)
panel_strict.to_csv(panel_strict_path, index=False)
schema_df.to_csv(schema_path, index=False)
missing_report.to_csv(missing_path, index=False)
diagnostics.to_csv(diagnostics_path, index=False)

panel_common_path, panel_strict_path

(PosixPath('../../data/processed/CY-Bench/wheat_IN_panel_full_year.csv'),
 PosixPath('../../data/processed/CY-Bench/wheat_IN_panel_strict.csv'))

In [24]:
# Quick sanity checks

print("panel_common shape:", panel_common.shape)
print("panel_strict shape:", panel_strict.shape)
print("years:", panel_common["harvest_year"].min(), "to", panel_common["harvest_year"].max())
print("adm_ids:", panel_common["adm_id"].nunique())
print("columns:", len(panel_common.columns))

panel_common.sample(min(5, len(panel_common)), random_state=42)

panel_common shape: (7552, 45)
panel_strict shape: (6715, 46)
years: 2001 to 2017
adm_ids: 502
columns: 45


,crop_name,country_code,adm_id,harvest_year,yield,production,harvest_area,sos,eos,crop_area,crop_area_percentage,latitude,longitude,region_area,awc,bulk_density,drainage_class,avg_tmin,avg_tmax,avg_tavg,avg_prec,sum_prec,avg_rad,avg_et0,avg_vpd,avg_cwb,n_meteo_obs,avg_ssm,min_ssm,max_ssm,avg_rsm,min_rsm,max_rsm,n_sm_obs,avg_ndvi,min_ndvi,max_ndvi,std_ndvi,n_ndvi_obs,avg_fpar,min_fpar,max_fpar,std_fpar,n_fpar_obs,n_available_blocks
1153,wheat,IN,IN-04-0177,2008,4.827,1077000.0,223100.0,345.775,105.806,3237.4,79.503,29.27363,75.79833,4072.0,15.526,1.445,6.0,18.569516,30.424620,24.118022,2.025025,741.159,1.870360e+07,4.542391,27.934658,-2.517366,366.0,4.124570,1.571,6.030,225.384044,205.555,272.726,365.0,0.404333,0.144,0.617,0.145582,45.0,39.881056,11.413,66.662,16.856514,36.0,4
393,wheat,IN,IN-02-0929,2006,1.601,131490.0,82150.0,344.495,102.920,82.0,2.343,26.35818,86.01512,3499.0,16.142,1.467,5.0,20.272800,29.915512,24.960348,4.165214,1520.303,1.782733e+07,4.191260,20.870501,-0.026047,365.0,4.668019,2.937,6.393,227.382871,172.837,292.868,365.0,0.428447,0.283,0.588,0.088247,38.0,38.179917,25.603,47.540,6.252077,36.0,4
742,wheat,IN,IN-03-0124,2002,0.897,15700.0,17500.0,336.982,102.269,97.8,1.983,21.87999,72.87551,4931.0,14.599,1.551,4.0,22.004148,33.344962,27.223592,1.913641,698.479,1.974153e+07,5.477981,33.070173,-3.564340,365.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.348000,0.234,0.466,0.075160,42.0,26.004278,15.196,35.338,7.356638,36.0,3
2879,wheat,IN,IN-07-0116,2002,1.057,18600.0,17600.0,329.958,89.760,695.9,5.115,20.11707,78.05961,13604.0,14.375,1.520,4.0,21.833833,32.505063,27.020200,2.981233,1088.150,1.913709e+07,5.111986,32.716041,-2.130753,365.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.396000,0.212,0.696,0.138225,42.0,32.733972,13.643,53.877,13.387089,36.0,3
23,wheat,IN,IN-01-0052,2006,0.263,50.0,190.0,314.176,101.958,118.1,0.608,14.87560,77.34445,19428.0,13.380,1.604,4.0,21.900611,32.214575,26.802614,1.534814,560.207,1.962574e+07,5.472184,29.515137,-3.937370,365.0,4.624860,1.902,7.124,273.062600,260.489,309.283,365.0,0.336283,0.250,0.566,0.078792,46.0,28.006861,17.525,45.293,9.180754,36.0,4
